# Linguistic Features Evaluation Notebook

## Mount Google Drive

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Step 1: Setup & Install Dependencies

In [2]:
%pip install transformers jsonlines tqdm

In [3]:
import torch
from transformers import RobertaTokenizer
import pandas as pd
from torch.utils.data import DataLoader
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import random
import joblib
import numpy as np
from torch.utils.data import Dataset
import jsonlines
from tqdm import tqdm
from transformers import RobertaModel

## Step 2: Load Resources

### 🔹 2.1 Load Gemini Features

In [4]:
# Load Gemini features
test_df = pd.read_csv("/content/drive/MyDrive/SemEval-2024 Task 8/Gemini Features/gemini_linguistic_extraction_monolingual.csv")

### 🔹 2.2 Preprocess Features

In [5]:
# Load scaler
scaler = joblib.load("/content/drive/MyDrive/SemEval-2024 Task 8/models/scaler.pkl")

In [ ]:
# Binary columns: convert and fill missing
binary_cols = ['HasParticipialPhrase', 'HasHangingParticiple', 'UsesPassiveVoice', 'SentenceStartRepetition']
test_df[binary_cols] = test_df[binary_cols].apply(pd.to_numeric, errors='coerce').fillna(0).astype(int)

# Numeric columns (only actual numeric features)
num_cols = [
    'NumIndependentClauses', 'NumDependentClauses', 'AvgSentenceLength',
    'SentenceLengthVariance', 'ClauseEmbeddingDepth',
    'TypeTokenRatio', 'NumIdioms'
]

# Normalize
test_df[num_cols] = scaler.fit_transform(test_df[num_cols])

# Drop the 'SentenceStructure' column
test_df = test_df.drop(columns=['Index', 'SentenceStructure'])

## Step 3: Define Gemini Text Dataset and the Model

### 🔹 3.1 Define Gemini Text Dataset Class

In [6]:
class GeminiTextDataset(torch.utils.data.Dataset):
    def __init__(self, jsonl_file, gemini_df, tokenizer, max_length=512):
        self.data = []
        self.max_length = max_length
        self.df = pd.read_json(jsonl_file, lines=True)
        self.features = gemini_df
        self.tokenizer = tokenizer

        # Load texts and labels
        with jsonlines.open(jsonl_file) as reader:
            entries = list(reader)

        assert len(entries) == len(gemini_df), "Mismatch between JSONL and CSV rows"

        for i in tqdm(range(len(entries)), desc=f"Loading {jsonl_file} with Gemini features"):
            obj = entries[i]
            text = obj["text"]
            label = obj["label"]

            # Tokenize text
            encoding = tokenizer(
                text,
                truncation=True,
                padding='max_length',
                max_length=max_length,
                return_tensors='pt'
            )

            # Get the corresponding Gemini features (as numpy array)
            feats = gemini_df.iloc[i].drop("Index", errors="ignore").values.astype(float)

            self.data.append((encoding, feats, label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        encoding, ling_feats, label = self.data[idx]
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'ling_feats': torch.tensor(ling_feats, dtype=torch.float),
            'labels': torch.tensor(label, dtype=torch.long)
        }

### 🔹 3.2 Define the Model

In [7]:
class RobertaWithGeminiFeatures(nn.Module):
    def __init__(self, ling_feat_dim, num_labels=2):
        super().__init__()
        self.roberta = RobertaModel.from_pretrained("roberta-base")
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(self.roberta.config.hidden_size + ling_feat_dim, num_labels)

    def forward(self, input_ids, attention_mask, ling_feats):
        outputs = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0]  # [CLS] token
        combined = torch.cat((cls_embedding, ling_feats), dim=1)
        combined = self.dropout(combined)
        logits = self.classifier(combined)
        return logits

## Step 4: Load the Test Dataset

### 🔹 4.1 Create the Test Dataset

In [ ]:
# Load tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

# Define test dataset path
test_file = "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_monolingual.jsonl"

# Create datasets
# test_dataset = GeminiTextDataset(test_file, test_df, tokenizer)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Loading /content/drive/MyDrive/SemEval-2024 Task 8/Datasets/subtaskA_monolingual.jsonl with Gemini features: 100%|██████████| 34272/34272 [03:35<00:00, 158.95it/s]


### 🔹 4.2 Save the Test Dataset

In [ ]:
# Save the datasets for the first time run
# torch.save(test_dataset, "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/pytorch datasets/test_dataset_ver2.pt")

### 🔹 4.3 Load the Test Dataset

In [8]:
test_dataset = torch.load(
    "/content/drive/MyDrive/SemEval-2024 Task 8/Datasets/pytorch datasets/test_dataset_ver2.pt",
    weights_only=False
)

# Create DataLoader
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

## Step 5: Load the Trained Model

In [9]:
model = RobertaWithGeminiFeatures(test_df.shape[1] - 2)
model.load_state_dict(torch.load("/content/drive/MyDrive/SemEval-2024 Task 8/models/linguistic models/linguistic_model_ver2.pth"))

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


<All keys matched successfully>

## Step 6: Evaluate the Model

In [10]:
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

all_preds, all_labels = [], []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        ling_feats = batch["ling_feats"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask, ling_feats)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Print classification report
print(classification_report(all_labels, all_preds, digits=4))

Evaluating: 100%|██████████| 2142/2142 [03:25<00:00, 10.41it/s]


              precision    recall  f1-score   support

           0     0.8071    0.7640    0.7849     16272
           1     0.7964    0.8349    0.8152     18000

    accuracy                         0.8012     34272
   macro avg     0.8018    0.7994    0.8001     34272
weighted avg     0.8015    0.8012    0.8009     34272

